In [ ]:
import os
import torch
import logging
import numpy as np
import netCDF4 as nc
from functools import partial
from typing import Any, Callable, Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import xarray as xr
import dask
import cartopy.crs as ccrs

In [ ]:
import torch

# Check if CUDA (NVIDIA GPU) is available
cuda_available = torch.cuda.is_available()

if cuda_available:
    print("CUDA (GPU) is available.")
    print("Number of CUDA devices:", torch.cuda.device_count())
    print("CUDA Device Name:", torch.cuda.get_device_name(0)) # Get name of the first GPU
else:
    print("No CUDA (GPU) available. PyTorch will use CPU.")

# Load dataset

In [ ]:
import xarray as xr

# Load dataset from Zarr
train_dataset = xr.open_zarr("../data/train_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)
val_dataset = xr.open_zarr("../data/val_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)
test_dataset  = xr.open_zarr("../data/test_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)
#test_dataset  = xr.open_zarr("/glade/work/asampath/cesm_backup/AS_val_dataset_normalized_by_5member_monthly_adjacent_timestep_TGCLDLWP.zarr", consolidated=True)
static_ds = xr.open_zarr("../data/static_input_no_poles.zarr", consolidated=False)
static_input_tensor = torch.tensor(static_ds["static_input"].values, dtype=torch.float32).unsqueeze(0)
static_repeated = static_input_tensor.repeat(2, 1, 1, 1)

In [ ]:
from torch.utils.data import Dataset
class XarrayBalancedBatchDataset_monthly(Dataset):
    def __init__(self, xarray_ds, shuffle=True):
        """
        Dataset that samples 10 label=0 and 10 label=1 samples
        for each (month, member) combination.
        """
        self.data = xarray_ds
        self.labels = self.data["labels"].values
        self.months = self.data["month"][:,0].values
        self.members = self.data["member"].values

        # Create list of unique (month, member) pairs
        month_member_pairs = list(zip(self.months, self.members))
        self.unique_pairs = list(set(month_member_pairs))

        # Shuffle for randomness
        if shuffle:
            np.random.seed(42)
            np.random.shuffle(self.unique_pairs)

    def __len__(self):
        return len(self.unique_pairs)

    def __getitem__(self, idx):
        """
        Returns 20 indices from the same (month, member), balanced by label.
        """
        month_value, member_value = self.unique_pairs[idx]

        # Filter for the current (month, member)
        mask = (self.months == month_value) & (self.members == member_value)

        # Separate by labels
        zero_indices = np.flatnonzero((self.labels == 0) & mask)
        one_indices = np.flatnonzero((self.labels == 1) & mask)
        selected_indices = np.concatenate([zero_indices, one_indices])
        return selected_indices

In [ ]:
# Full version of XarrayBalancedBatchPrefetcher compatible with BatchForecastLenDataLoader

from torch.utils.data import Dataset
from multiprocessing import Process, Queue, Event, Manager
from threading import Thread
import numpy as np
import torch
import time
import signal

class XarrayBalancedBatchPrefetcher(Dataset):
    def __init__(self, xarray_ds, static_repeated, shuffle=True, num_workers=4, prefetch_factor=8):
        self.data = xarray_ds
        self.labels = self.data["labels"].values
        self.months = self.data["month"].values[:, 0]  # use month[:, 0] as base time
        self.members = self.data["member"].values
        self.static_repeated = static_repeated

        all_pairs = list(set(zip(self.months, self.members)))
        all_pairs.sort()
        self.unique_pairs = all_pairs
        print(len(self.unique_pairs))
        if shuffle:
            np.random.seed(42)
            np.random.shuffle(self.unique_pairs)

        self.num_workers = num_workers
        self.prefetch_factor = prefetch_factor
        self.prefetch_queue = Queue(maxsize=prefetch_factor)
        self.stop_event = Event()
        self.manager = Manager()
        self.results = self.manager.dict()

        signal.signal(signal.SIGTERM, self.handle_signal)

        self.prefetch_thread = Thread(target=self.prefetch_batches, daemon=True)
        self.prefetch_thread.start()

    def set_epoch(self, epoch):
        print(f"set_epoch({epoch}) called")

        self.stop_event.set()
        if hasattr(self, "prefetch_thread") and self.prefetch_thread is not None:
            if self.prefetch_thread.is_alive():
                self.prefetch_thread.join(timeout=5)

        self.stop_event = Event()
        self.prefetch_queue = Queue(maxsize=self.prefetch_factor)

        try:
            if hasattr(self, "manager") and self.manager is not None:
                self.manager.shutdown()
        except Exception as e:
            print(f"Warning: failed to shutdown manager: {e}")

        self.manager = Manager()
        self.results = self.manager.dict()

        self.prefetch_thread = Thread(target=self.prefetch_batches, daemon=True)
        self.prefetch_thread.start()

    def __len__(self):
        return len(self.unique_pairs)

    def __getitem__(self, _):
        #return self.prefetch_queue.get()
        try:
                return self.prefetch_queue.get(timeout=5)  # wait up to 5 sec
        except Exception as e:
            print("Prefetch queue timeout:", e)
            raise StopIteration  # signal end of data

    def handle_signal(self, signum, frame):
        print("Received termination signal. Stopping prefetching...")
        self.stop_event.set()

    def prefetch_batches(self):
        idx = 0
        try:
            while not self.stop_event.is_set():
                if not self.prefetch_queue.full():
                    chunk = self.unique_pairs[idx:idx + self.num_workers]
                    if self.results is None:
                        if self.manager is None:
                            self.manager = Manager()
                        self.results = self.manager.dict()
                    else:
                        self.results.clear()
                    processes = [
                        Process(target=self.worker_process, args=(i, pair, self.results))
                        for i, pair in enumerate(chunk)
                    ]
                    for p in processes:
                        p.start()
                    for p in processes:
                        p.join()

                    for i in range(len(chunk)):
                        if self.results.get(i) is not None:
                            self.prefetch_queue.put(self.results[i])
                    idx += self.num_workers
                else:
                    time.sleep(0.05)
        except Exception as e:
            print(f"Prefetching error: {e}")
    '''
    def worker_process(self, k, pair, result_dict):
        try:
            month_value, member_value = pair
            mask = (self.months == month_value) & (self.members == member_value)
            zero_indices = np.flatnonzero((self.labels == 0) & mask)
            one_indices = np.flatnonzero((self.labels == 1) & mask)

            selected_indices = np.concatenate([zero_indices, one_indices])
            if len(selected_indices) == 0:
                result_dict[k] = None
                return

            data_batch = self.data["data"].isel(pairs=selected_indices).load()
            x1 = torch.from_numpy(data_batch[:, 0].values).float()
            x2 = torch.from_numpy(data_batch[:, 1].values).float()

            x1_combined = torch.cat([x1, self.static_repeated], dim=1)
            x2_combined = torch.cat([x2, self.static_repeated], dim=1)

            result_dict[k] = {"x1": x1_combined, "x2": x2_combined}

        except Exception as e:
            print(f"Worker error for pair {pair}: {e}")
            result_dict[k] = None
    '''
    def worker_process(self, k, pair, result_dict):
        try:
            month_value, member_value = pair
            mask = (self.months == month_value) & (self.members == member_value)
            zero_indices = np.flatnonzero((self.labels == 0) & mask)
            one_indices = np.flatnonzero((self.labels == 1) & mask)

            selected_indices = np.concatenate([zero_indices, one_indices])
            if len(selected_indices) == 0:
                result_dict[k] = None
                return

            data_batch = self.data["data"].isel(pairs=selected_indices).load()
            label_batch = torch.tensor(self.labels[selected_indices], dtype=torch.long)

            x1 = torch.from_numpy(data_batch[:, 0].values).float()
            x2 = torch.from_numpy(data_batch[:, 1].values).float()

            x1_combined = torch.cat([x1, self.static_repeated], dim=1)
            x2_combined = torch.cat([x2, self.static_repeated], dim=1)
            
            member=self.data.member.isel(pairs=selected_indices).load()

            result_dict[k] = {
                "x1": x1_combined,
                "x2": x2_combined,
                "x1_labels": label_batch,
                "x2_labels": label_batch,
                "member":member.values.tolist(),
                "month":month_value
            }

        except Exception as e:
            print(f"Worker error for pair {pair}: {e}")
            result_dict[k] = None
        
    def __del__(self):
        try:
            print("Cleaning up...")
            self.stop_event.set()
            if hasattr(self, "prefetch_thread") and self.prefetch_thread.is_alive():
                self.prefetch_thread.join(timeout=5)
            if hasattr(self, "manager"):
                try:
                    self.manager.shutdown()
                except Exception as e:
                    print(f"Manager shutdown error: {e}")
            if hasattr(self, "prefetch_queue"):
                while not self.prefetch_queue.empty():
                    try:
                        self.prefetch_queue.get_nowait()
                    except Exception:
                        break
            print("Cleanup complete.")
        except Exception as e:
            print(f"Cleanup error: {e}")

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.__del__()

    @property
    def forecast_len(self):
        return 0

    def batches_per_epoch(self):
        return len(self)


class BatchForecastLenDataLoader:
    def __init__(self, dataset):
        self.dataset = dataset
        self.forecast_len = dataset.forecast_len + 1

    '''
    def __iter__(self):
        dataset_iter = iter(self.dataset)
        for _ in range(len(self)):
            yield next(dataset_iter)
    '''
    def __iter__(self):
        dataset_iter = iter(self.dataset)
        max_batches = len(self)  # total batches expected
        for _ in range(max_batches):
            try:
                yield next(dataset_iter)
            except StopIteration:
                break

    def __len__(self):
        if hasattr(self.dataset, "batches_per_epoch"):
            return self.dataset.batches_per_epoch() * self.forecast_len
        else:
            return len(self.dataset) * self.forecast_len


In [ ]:
import torch
import xarray as xr


# Load and prepare static input
static_ds = xr.open_zarr("../data/static_input_no_poles.zarr", consolidated=False)
static_input_tensor = torch.tensor(static_ds["static_input"].values, dtype=torch.float32).unsqueeze(0)
static_repeated = static_input_tensor.repeat(2, 1, 1, 1)

# Initialize datasets
train_dataset = XarrayBalancedBatchPrefetcher(
    xarray_ds=train_dataset,
    static_repeated=static_repeated,
    shuffle=True,
    num_workers=8,
    prefetch_factor=8,
)

val_dataset = XarrayBalancedBatchPrefetcher(
    xarray_ds=val_dataset,
    static_repeated=static_repeated,
    shuffle=False,
    num_workers=8,
    prefetch_factor=8,
)
test_dataset = XarrayBalancedBatchPrefetcher(
    xarray_ds=test_dataset,
    static_repeated=static_repeated,
    shuffle=False,
    num_workers=8,
    prefetch_factor=8,
)

# Wrap in batch loaders
#train_loader = BatchForecastLenDataLoader(train_dataset)
#val_loader = BatchForecastLenDataLoader(val_dataset)
test_loader = BatchForecastLenDataLoader(test_dataset)


# Evalution

### For VGG-GELU-No-AdaptivePooling

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import torchvision.models as models

# Detect if CUDA is available
seed = 1#3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "../model/contrastive_model_vgg_seed_42_rank_0_100_accumulation_steps_monthly_radiation_LWP_no_poles.pth"
#"/glade/u/home/asampath/DF_copy_Climatesleuth_github/miles-climatesleuth/scripts/AS_contrastive_model_vgg_seed_42_rank_0_4_accumulation_steps_GELU_TGCLDLWP.pth"

class MeanPooledVGG(nn.Module):
    def __init__(self, in_channels=5, projection_dim=128):
        super(MeanPooledVGG, self).__init__()

        # Feature extractor using only Conv2d and mean pooling
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.GELU(), 

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GELU(), 
            nn.AvgPool2d(kernel_size=4,stride=4),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.GELU(), 
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.GELU(), 
            nn.AvgPool2d(kernel_size=4,stride=4),
        )

        # Projector head
        self.projector = nn.Sequential(
            nn.Flatten(),              # (B, 512, 1, 1) -> (B, 512)
            nn.Linear(36864, 2048),
            nn.GELU(),
            nn.Linear(2048, 256),
            nn.GELU(),
            #nn.ReLU(inplace=True),
            nn.Linear(256, projection_dim)
        )

    def forward(self, x):
        features = self.backbone(x)
        projection = self.projector(features)
        return projection


# Example usage
model = MeanPooledVGG(in_channels=9, projection_dim=128).to(device)
#model.load_state_dict(torch.load(save_path))
state_dict = torch.load(save_path)

# Remove 'module.' prefix from all keys
new_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

model.load_state_dict(new_state_dict)

# Set model to evaluation mode
model.eval()


In [ ]:
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm


# Put model in eval mode
model.eval()

def extract_features(model, dataloader, device):
    features = []
    labels = []
    members = []
    months = []

    model.to(device)
    model.eval()

    with torch.no_grad():
        for batch in tqdm(dataloader, total=len(dataloader), desc="Extracting features"):
            x1 = batch["x1"].to(device, non_blocking=True)         # [N, C, H, W]
            #x2 = batch["x2"].to(device, non_blocking=True)         # [N, C, H, W]
            labels1 = batch["x1_labels"].cpu().numpy()             # [N]
            #labels2 = batch["x2_labels"].cpu().numpy()             # [N]
            member = np.asarray(batch["member"])#to(device, non_blocking=True)          # [N, C, H, W]
            #member = batch.member.isel(pairs=batch_indices).values
            month = np.asarray(batch["month"])

            # Feature extraction
            z1 = model(x1).cpu().numpy()
            #z2 = model(x2).cpu().numpy()

            # Mean LWP proxy
            f1 = x1.mean(dim=(1, 2, 3)).cpu().numpy()
            #f2 = x2.mean(dim=(1, 2, 3)).cpu().numpy()

            # Combine all
            features.extend(z1)
            #features.extend(z2)
            labels.extend(labels1)
            #labels.extend(labels2)
            members.extend(member)
            #members.extend(member)
            #print(month)
            months.extend([month])
            months.extend([month])

    return np.array(features), np.array(labels), np.array(members), np.array(months)


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract features
X_train, y_train, train_members, train_months = extract_features(model, train_loader, device)
X_val, y_val, val_members, val_months = extract_features(model, val_loader, device)
X_test, y_test, test_members, test_months = extract_features(model, test_loader, device)

# Train linear classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=["ML (Class 0)", "KK (Class 1)"])

# Output
print(f"Linear Evaluation Accuracy: {accuracy:.4f}")
print("Classification Report:\n", report)


In [ ]:
X_val, y_val, val_members, val_months = extract_features(model, val_loader, device)
X_test, y_test, test_members, test_months = extract_features(model, test_loader, device)

In [ ]:
import xarray as xr
import numpy as np

# Convert data to xarray Dataset
ds = xr.Dataset({
    "X_train": (["sample", "feature"], np.array(X_train)),
    "y_train": (["sample"], np.array(y_train)),
    "train_members": (["sample"], np.array(train_members)),
    "train_months": (["sample"], np.array(train_months)),

    "X_val": (["val_sample", "feature"], np.array(X_val)),
    "y_val": (["val_sample"], np.array(y_val)),
    "val_members": (["val_sample"], np.array(val_members)),
    "val_months": (["val_sample"], np.array(val_months)),

    "X_test": (["test_sample", "feature"], np.array(X_test)),
    "y_test": (["test_sample"], np.array(y_test)),
    "test_members": (["test_sample"], np.array(test_members)),
    "test_months": (["test_sample"], np.array(test_months)),
})
# Save to Zarr file
ds.to_zarr("../data/latent_features_VGG_monthly_radiation_LWP_no_poles.zarr", mode="w")

In [ ]:
import xarray as xr
import numpy as np

# Load Zarr dataset
ds = xr.open_zarr("../data/latent_features_VGG_monthly_radiation_LWP_no_poles.zarr", consolidated=False)

# Extract numpy arrays
X_train = ds["X_train"].values
y_train = ds["y_train"].values
train_members = ds["train_members"].values
train_months = ds["train_months"].values

X_val = ds["X_val"].values
y_val = ds["y_val"].values
val_members = ds["val_members"].values
val_months = ds["val_months"].values

X_test = ds["X_test"].values
y_test = ds["y_test"].values
test_members = ds["test_members"].values
test_months = ds["test_months"].values


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

plt.rcParams['font.size'] = 20

# ---------- L2-normalize embeddings ----------
def l2_normalize(X, axis=1, eps=1e-8):
    """Normalize each sample vector to unit length."""
    norm = np.linalg.norm(X, axis=axis, keepdims=True)
    return X / (norm + eps)

# ---------- Compute mean TAU–KK cosine distances per member ----------
def compute_mean_cosine_distances(X, y, member_ids):
    # Normalize once globally
    X = l2_normalize(X)

    unique_members = np.unique(member_ids)
    mean_distances = []

    for member in unique_members:
        mask = (member_ids == member)
        X_member = X[mask]
        y_member = y[mask]
        tau = X_member[y_member == 0]
        kk = X_member[y_member == 1]

        if len(tau) > 0 and len(kk) > 0:
            # cdist with 'cosine' returns 1 - cos_sim by definition
            dist = cdist(tau, kk, metric='cosine')
            mean_distances.append(np.mean(dist))
        else:
            mean_distances.append(np.nan)
    
    return unique_members, mean_distances

# ---------- Compute mean distances ----------
train_ids, train_dists = compute_mean_cosine_distances(X_train, y_train, train_members)
val_ids, val_dists = compute_mean_cosine_distances(X_val, y_val, val_members)
test_ids, test_dists = compute_mean_cosine_distances(X_test, y_test, test_members)

# ---------- Plot ----------
plt.figure(figsize=(14, 6))
plt.plot(train_ids, train_dists, 'o-', label='Train')
plt.plot(val_ids, val_dists, 's-', label='Validation')
plt.plot(test_ids, test_dists, '^-', label='Test')

# Vertical split markers
plt.axvline(x=60.5, color='gray', linestyle='--', label='Train/Val Split')
plt.axvline(x=80.5, color='gray', linestyle='--', label='Val/Test Split')

plt.xlabel("Member Number")
plt.ylabel("Mean Cosine Distance (1 − cos sim)")
plt.title("Mean Latent Cosine Distance Between TAU and KK Per Member (normalized latent space)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()


## latent space visualization:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

plt.rcParams['font.size'] = 20

def visualize_features_3d(
    X, y,                     # test set (backward compat)
    method="pca",
    X_obs=None, y_obs=None,   # observations (label 2)
    X_train=None, y_train=None,
    X_val=None, y_val=None,
):
    """
    Visualizes features in 3D (PCA or t-SNE) with *class-based* styling:
      - TAU-ML (0): blue hollow circles
      - KK2000 (1): red  hollow circles
      - OBS   (2): green filled triangles

    Returns:
        X_test_3d, X_obs_3d, explained_ratio, X_train_3d, X_val_3d
    """
    # ensure arrays
    X = np.asarray(X); y = np.asarray(y)

    has_obs   = (X_obs   is not None) and (y_obs   is not None)
    has_train = (X_train is not None) and (y_train is not None)
    has_val   = (X_val   is not None) and (y_val   is not None)

    # collect splits in order to slice back later
    splits = [("test",  X, y)]
    if has_obs:   splits.append(("obs",   np.asarray(X_obs),   np.asarray(y_obs)))
    if has_train: splits.append(("train", np.asarray(X_train), np.asarray(y_train)))
    if has_val:   splits.append(("val",   np.asarray(X_val),   np.asarray(y_val)))

    sizes = {name: Xs.shape[0] for name, Xs, _ in splits}
    X_all = np.vstack([Xs for _, Xs, _ in splits])
    y_all = np.concatenate([ys for _, _, ys in splits])

    # normalize jointly
    X_norm = StandardScaler().fit_transform(X_all)

    # reduce to 3D
    explained_ratio = None
    if method.lower() == "tsne":
        n = X_norm.shape[0]
        perplexity = max(5, min(30, (n - 1) // 3))
        print(f"Performing 3D t-SNE (perplexity={perplexity})...")
        reducer = TSNE(n_components=3, perplexity=perplexity,
                       learning_rate='auto', init='pca', random_state=42)
        X_red = reducer.fit_transform(X_norm)
    elif method.lower() == "pca":
        print("Performing 3D PCA...")
        pca = PCA(n_components=3)
        X_red = pca.fit_transform(X_norm)
        explained_ratio = pca.explained_variance_ratio_
        print("Explained variance ratio (3D):", explained_ratio)
    else:
        raise ValueError("Method should be either 'tsne' or 'pca'")

    # slice back out
    offsets, start = {}, 0
    for name in [n for n, _, _ in splits]:
        nrows = sizes[name]
        offsets[name] = (start, start + nrows)
        start += nrows

    def slice_split(name):
        if name not in offsets: return None
        a, b = offsets[name]
        return X_red[a:b, :]

    X_test_3d  = slice_split("test")
    X_obs_3d   = slice_split("obs")
    X_train_3d = slice_split("train")
    X_val_3d   = slice_split("val")

    # ---------------- plot ----------------
    fig = plt.figure(figsize=(13, 11))
    ax = fig.add_subplot(111, projection='3d')

    class_names = {0: "TAU-ML", 1: "KK2000", 2: "OBS"}
    # same color across splits for each class
    class_styles = {
        0: dict(edgecolors="blue",  facecolors='none', marker='o', s=60, linewidths=1.2),
        1: dict(edgecolors="red",   facecolors='none', marker='o', s=60, linewidths=1.2),
        2: dict(edgecolors="black", facecolors='tab:green', marker='^', s=70, alpha=0.9),
    }

    # Plot all points by class (train/val/test combined for 0/1; obs separately as 2)
    for lbl in [0, 1, 2]:
        m = (y_all == lbl)
        if np.any(m):
            st = class_styles[lbl]
            ax.scatter(X_red[m, 0], X_red[m, 1], X_red[m, 2],
                       label=class_names[lbl], **st)

    # title
    title_bits = []
    if has_train: title_bits.append("train")
    if has_val:   title_bits.append("val")
    title_bits.append("test")
    if has_obs:   title_bits.append("obs")
    title_suffix = "(" + " + ".join(title_bits) + ")"

    ax.set_title(f"3D Latent Space using {method.upper()} {title_suffix}")
    ax.set_xlabel("Dim 1"); ax.set_ylabel("Dim 2"); ax.set_zlabel("Dim 3")
    ax.legend(loc="best", frameon=True)
    plt.tight_layout()
    plt.show()

    return X_test_3d, X_obs_3d, explained_ratio, X_train_3d, X_val_3d

# Example:
X_test_3d, X_obs_3d, explained_ratio, X_train_3d, X_val_3d = visualize_features_3d(
    X_test, y_test, method="pca",
    X_obs=X_obs, y_obs=y_obs,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.rcParams['font.size'] = 16

def visualize_all_pc_combinations(X, y):
    """
    Plots all 2D combinations of PC1, PC2, and PC3 for latent vectors.
    """
    # Normalize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Fit PCA
    print("Performing PCA...")
    pca = PCA(n_components=3)
    X_pca = pca.fit_transform(X_scaled)
    explained_ratio = pca.explained_variance_ratio_
    print("Explained variance ratio:", explained_ratio)

    pc_labels = [f"PC{i+1} ({explained_ratio[i]:.2%})" for i in range(3)]
    combinations = [(0, 1), (0, 2), (1, 2)]  # (PC1 vs PC2), (PC1 vs PC3), (PC2 vs PC3)

    class_labels = {0: "TAU-ML", 1: "KK2000"}
    colors = ["blue", "red"]

    # Plot all combinations
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for ax, (i, j) in zip(axes, combinations):
        for label, color in zip(class_labels.keys(), colors):
            mask = (y == label)
            ax.scatter(
                X_pca[mask, i], X_pca[mask, j],
                edgecolors=color, facecolors='none', s=60, label=class_labels[label]
            )
        ax.set_xlabel(pc_labels[i])
        ax.set_ylabel(pc_labels[j])
        ax.set_title(f"{pc_labels[i]} vs {pc_labels[j]}")
        ax.grid(True, linestyle='--', alpha=0.5)
    
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right', fontsize=14)
    fig.suptitle("2D PCA Projections of Latent Space", fontsize=18)
    plt.tight_layout(rect=[0, 0, 0.95, 0.95])
    plt.show()

    return X_pca, explained_ratio

#X_pca, explained_ratio = visualize_all_pc_combinations(,X_test, y_test)
X_pca, explained_ratio = visualize_all_pc_combinations(np.concatenate((X_train, X_val, X_test)), np.concatenate((y_train, y_val, y_test)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe

plt.rcParams.update({'font.size': 16})

def _mm_strings(months):
    m = np.asarray(months)
    if m.dtype.kind in "OUS":  # strings/objects
        return np.array([str(x)[-2:] for x in m])
    return np.array([f"{int(x):02d}" for x in m])  # integers -> "01".."12"

def _make_norm(vals):
    vals = np.asarray(vals)
    if vals.size == 0:
        return plt.Normalize(0, 1)
    vmin, vmax = np.min(vals), np.max(vals)
    if vmin == vmax:
        vmin, vmax = vmin - 0.5, vmax + 0.5
    return plt.Normalize(vmin, vmax)

def visualize_features_3d_by_month(
    X_3d, y, months, member_ids,
    X_obs_3d=None, y_obs=None, obs_months=None, data_type_obs=None,
    obs_obstype_value=1,
    axis_lim=15
):
    """
    Monthly 3D panels with TAU-ML (blue), KK2000 (red), and OBS (green triangles).
    Only observations with obstype == obs_obstype_value are shown.

    Args:
      X_3d, y, months, member_ids : model/test embeddings & metadata
      X_obs_3d, y_obs, obs_months, data_type_obs : observation embeddings & metadata
      obs_obstype_value : keep only obs where data_type_obs == this value (default 1)
      axis_lim : sets x/y/z limits to [-axis_lim, axis_lim]
    """
    # --- month handling ---
    cal_m = _mm_strings(months)

    # filter obs: only label==2 and obstype==obs_obstype_value
    have_obs = (
        X_obs_3d is not None and y_obs is not None and
        obs_months is not None and data_type_obs is not None
    )
    if have_obs:
        y_obs = np.asarray(y_obs)
        data_type_obs = np.asarray(data_type_obs)
        keep = (y_obs == 2) & (data_type_obs == obs_obstype_value)
        X_obs_3d = np.asarray(X_obs_3d)[keep]
        obs_m = _mm_strings(np.asarray(obs_months)[keep])
    else:
        X_obs_3d = None
        obs_m = None

    # --- figure & colormaps ---
    fig = plt.figure(figsize=(24, 16), dpi=500)
    gs = gridspec.GridSpec(3, 6, width_ratios=[0.2, 1, 1, 1, 1, 0.2])

    cmap_tau, cmap_kk = plt.cm.Blues, plt.cm.Reds
    norm_tau = _make_norm(np.asarray(member_ids)[y == 0])
    norm_kk  = _make_norm(np.asarray(member_ids)[y == 1])

    month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

    # to avoid legends in every panel
    legend_done = False

    for i, mm in enumerate([f"{k:02d}" for k in range(1, 13)]):
        ax = fig.add_subplot(gs[i // 4, (i % 4) + 1], projection='3d')

        m = (cal_m == mm)
        X_m, y_m, mem_m = X_3d[m], y[m], np.asarray(member_ids)[m]

        tau_mask = (y_m == 0)
        kk_mask  = (y_m == 1)

        # TAU-ML: blue hollow circles
        if tau_mask.any():
            ax.scatter(X_m[tau_mask, 0], X_m[tau_mask, 1], X_m[tau_mask, 2],
                       c=cmap_tau(norm_tau(mem_m[tau_mask])),
                       edgecolors=cmap_tau(norm_tau(mem_m[tau_mask])),
                       s=50, marker='o', facecolors='none',
                       label=None if legend_done else "TAU-ML",  alpha=0.5, depthshade=True)

        # KK2000: red filled circles
        if kk_mask.any():
            ax.scatter(X_m[kk_mask, 0], X_m[kk_mask, 1], X_m[kk_mask, 2],
                       c=cmap_kk(norm_kk(mem_m[kk_mask])),
                       s=50, marker='o',
                       label=None if legend_done else "KK2000", alpha=0.5, depthshade=True)

        # OBS (obstype=1): green triangles
        if X_obs_3d is not None:
            mo = (obs_m == mm)
            if mo.any():
                '''
                ax.scatter(X_obs_3d[mo, 0], X_obs_3d[mo, 1], X_obs_3d[mo, 2],
                           c='tab:green', edgecolors='black', s=70, marker='^', alpha=0.95,
                           label=None if legend_done else "OBS (obstype=1)")
                '''
                obs_pts = ax.scatter(
                            X_obs_3d[mo, 0], X_obs_3d[mo, 1], X_obs_3d[mo, 2],
                            marker='^', s=200, facecolors='gold', edgecolors='black',
                            label=None if legend_done else "OBS (best estimate)",
                            linewidths=2.0, alpha=1.0, depthshade=False, zorder=500
                        )
                # add a white halo so it stays visible even over dark backgrounds
                obs_pts.set_path_effects([pe.withStroke(linewidth=3.5, foreground='white')])

        ax.set_title(month_labels[int(mm) - 1])
        ax.set_xlabel("Dim 1"); ax.set_ylabel("Dim 2"); ax.set_zlabel("Dim 3")
        #ax.set_xlim(-axis_lim, axis_lim); ax.set_ylim(-axis_lim, axis_lim); ax.set_zlim(-axis_lim, axis_lim)
        ax.set_xlim(-15, 15)
        ax.set_ylim(-15, 15)
        ax.set_zlim(-15, 15)

        if not legend_done:
            ax.legend(loc="upper center", bbox_to_anchor=(4.3, 1.6))
            legend_done = True

    # Left colorbar (TAU-ML members)
    cax1 = fig.add_subplot(gs[:, 0])
    sm1 = plt.cm.ScalarMappable(cmap=cmap_tau, norm=norm_tau); sm1.set_array([])
    cbar1 = fig.colorbar(sm1, cax=cax1); cbar1.set_label("TAU-ML Member #")

    # Right colorbar (KK2000 members)
    cax2 = fig.add_subplot(gs[:, -1])
    sm2 = plt.cm.ScalarMappable(cmap=cmap_kk, norm=norm_kk); sm2.set_array([])
    cbar2 = fig.colorbar(sm2, cax=cax2); cbar2.set_label("KK2000 Member #")

    fig.suptitle("3D Latent Space by Calendar Month (with monthly obs climatology)", fontsize=25, y=1.02)
    plt.subplots_adjust(top=0.92)
    plt.tight_layout()
    plt.show()

'''
visualize_features_3d_by_month(
    X_3d, y_test, test_months, test_members,
    X_obs_3d=X_obs_3d, y_obs=y_obs, obs_months=obs_months, data_type_obs=data_type_obs,
    obs_obstype_value=1,  # keep only obstype==1
    axis_lim=15
)
'''
visualize_features_3d_by_month(
    np.concatenate((X_train_3d, X_val_3d, X_test_3d)), np.concatenate((y_train, y_val, y_test)),
    np.concatenate((train_months, val_months, test_months)), np.concatenate((train_members, val_members, test_members)),
    X_obs_3d=X_obs_3d, y_obs=y_obs, obs_months=obs_months, data_type_obs=data_type_obs,
    obs_obstype_value=1,  # keep only obstype==1
    axis_lim=15
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import matplotlib.gridspec as gridspec
from matplotlib import cm
from matplotlib import patheffects as pe
from sklearn.preprocessing import StandardScaler
import matplotlib.animation as animation
import matplotlib.lines as mlines
import matplotlib.colors as mcolors

def truncate_colormap(cmap, minval=0.2, maxval=1.0, n=256):
    """Return a truncated copy of a colormap"""
    new_cmap = mcolors.LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

def _mm_strings(months):
    m = np.asarray(months)
    if m.dtype.kind in "OUS":
        return np.array([str(x)[-2:] for x in m])
    return np.array([f"{int(x):02d}" for x in m])

def _make_norm(vals):
    vals = np.asarray(vals)
    if vals.size == 0:
        return plt.Normalize(0, 1)
    vmin, vmax = np.min(vals), np.max(vals)
    if vmin == vmax:
        vmin, vmax = vmin - 0.5, vmax + 0.5
    return plt.Normalize(vmin, vmax)

def create_monthly_animation(
    X_3d, y, months, member_ids,
    filename="../figures/latent_space_by_month_with_colorbars_radiation_LWP_obs_no_poles.gif",
    X_obs_3d=None, y_obs=None, obs_months=None, data_type_obs=None, obs_obstype_value=1,
    axis_lim=15, fps=1
):
    calendar_months = _mm_strings(months)
    month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

    # filter observations: only label==2 and obstype==1 (best estimate)
    have_obs = (X_obs_3d is not None and y_obs is not None and
                obs_months is not None and data_type_obs is not None)
    if have_obs:
        y_obs = np.asarray(y_obs)
        data_type_obs = np.asarray(data_type_obs)
        keep = (y_obs == 2) & (data_type_obs == obs_obstype_value)
        X_obs_3d = np.asarray(X_obs_3d)[keep]
        obs_m = _mm_strings(np.asarray(obs_months)[keep])
    else:
        X_obs_3d = None
        obs_m = None

    fig = plt.figure(figsize=(12, 9))
    gs = gridspec.GridSpec(1, 3, width_ratios=[0.05, 0.9, 0.05])
    ax = fig.add_subplot(gs[1], projection='3d')

    cmap_tau = truncate_colormap(plt.cm.Blues, 0.2, 1.0)
    cmap_kk  = truncate_colormap(plt.cm.Reds,  0.2, 1.0)
    norm_tau = _make_norm(np.asarray(member_ids)[y == 0])
    norm_kk  = _make_norm(np.asarray(member_ids)[y == 1])

    def update(frame):
        mm = f"{frame + 1:02d}"
        ax.clear()
        ax.set_title(f"{month_labels[frame]}", fontsize=25)
        mask = (calendar_months == mm)
        X_m = X_3d[mask]
        y_m = y[mask]
        mem_m = np.asarray(member_ids)[mask]

        tau_mask = (y_m == 0)
        kk_mask  = (y_m == 1)
        
        # TAU-ML: blue hollow, slight transparency
        if tau_mask.any():
            ax.scatter(
                X_m[tau_mask, 0], X_m[tau_mask, 1], X_m[tau_mask, 2],
                c=cmap_tau(norm_tau(mem_m[tau_mask])),
                #edgecolors=cmap_tau(norm_tau(mem_m[tau_mask])),
                s=50, marker='o', alpha=0.3, depthshade=True, label='TAU-ML',zorder=1
            )

        # KK2000: red filled
        if kk_mask.any():
            ax.scatter(
                X_m[kk_mask, 0], X_m[kk_mask, 1], X_m[kk_mask, 2],
                c=cmap_kk(norm_kk(mem_m[kk_mask])), 
                s=50, marker='o', alpha=0.3, depthshade=True, label='KK2000',zorder=2
            )
        # OBS (best estimate): big gold triangle ON TOP
        if X_obs_3d is not None:
            mo = (obs_m == mm)
            if mo.any():
                pts = ax.scatter(
                    X_obs_3d[mo, 0], X_obs_3d[mo, 1], X_obs_3d[mo, 2],
                    marker='^', s=250, facecolors='gold', edgecolors='black',
                    linewidths=2.0, alpha=1.0, depthshade=False, zorder=10000,
                    label='OBS (best estimate)'
                )
                # white halo so it never gets lost
                #pts.set_path_effects([pe.withStroke(linewidth=3.5, foreground='white')])

        ax.set_xlim(-axis_lim, axis_lim)
        ax.set_ylim(-axis_lim, axis_lim)
        ax.set_zlim(-axis_lim, axis_lim)
        ax.set_xlabel("Dim 1"); ax.set_ylabel("Dim 2"); ax.set_zlabel("Dim 3")
        #ax.legend(loc='upper left')
        tau_handle = mlines.Line2D([], [], color=cmap_tau(0.6), marker='o',
                                   markersize=8, linestyle='None',
                                   markerfacecolor=cmap_tau(0.6), label='TAU-ML')
        kk_handle = mlines.Line2D([], [], color=cmap_kk(0.6), marker='o',
                                  markersize=8, linestyle='None',
                                  markerfacecolor=cmap_kk(0.6), label='KK2000')
        obs_handle = mlines.Line2D([], [], color='black', marker='^',
                                   markersize=10, linestyle='None',
                                   markerfacecolor='gold', label='OBS (best estimate)')

        ax.legend(handles=[tau_handle, kk_handle, obs_handle],
                  loc='upper left', frameon=True)
        

    # Colorbars (fixed outside the animation frames)
    cax1 = fig.add_subplot(gs[0])
    sm1 = plt.cm.ScalarMappable(cmap=cmap_tau, norm=norm_tau); sm1.set_array([])
    cbar1 = plt.colorbar(sm1, cax=cax1); cbar1.set_label("TAU-ML Member #")

    cax2 = fig.add_subplot(gs[2])
    sm2 = plt.cm.ScalarMappable(cmap=cmap_kk, norm=norm_kk); sm2.set_array([])
    cbar2 = plt.colorbar(sm2, cax=cax2); cbar2.set_label("KK2000 Member #")

    ani = animation.FuncAnimation(fig, update, frames=12, interval=1000)
    ani.save(filename, writer='pillow', fps=fps)
    plt.close()

# Example:
'''
create_monthly_animation(
    X_3d, y_test, test_months, test_members,
    X_obs_3d=X_obs_3d, y_obs=y_obs, obs_months=obs_months, data_type_obs=data_type_obs,
    obs_obstype_value=1, axis_lim=15
)
'''
create_monthly_animation(
    np.concatenate((X_train_3d,X_val_3d,X_test_3d)), np.concatenate((y_train,y_val,y_test)), np.concatenate((train_months, val_months, test_months)), 
    np.concatenate((train_members, val_members, test_members)),
    X_obs_3d=X_obs_3d, y_obs=y_obs, obs_months=obs_months, data_type_obs=data_type_obs,
    obs_obstype_value=1, axis_lim=15
)